# ✅ Solution: Debug Your Synthetic Dataset

This notebook contains our walkthrough of the challenge from Video 7. We identified three issues in the synthetic credit dataset and propose a concrete fix for each one. Compare it against your own analysis before reading on.

## 1. Load the Datasets

In [ ]:
import pandas as pd

original_df  = pd.read_csv('../data/credit_original.csv')
synthetic_df = pd.read_csv('../data/credit_synthetic.csv')

print(f'Original : {original_df.shape[0]:,} rows × {original_df.shape[1]} columns')
print(f'Synthetic: {synthetic_df.shape[0]:,} rows × {synthetic_df.shape[1]} columns')

## 2. Profiling Report

In [ ]:
from ydata_profiling import ProfileReport

original_profile  = ProfileReport(original_df,  title='Original',  minimal=True)
synthetic_profile = ProfileReport(synthetic_df, title='Synthetic', minimal=True)

comparison = original_profile.compare(synthetic_profile)
comparison.to_file('output/credit_profiling_comparison.html')
print('Profiling report saved → output/credit_profiling_comparison.html')
comparison

## 3. Quality Report

In [ ]:
from ydata.dataset import Dataset
from ydata.metadata import Metadata
from ydata.report import SyntheticDataProfile

original_dataset  = Dataset(original_df)
synthetic_dataset = Dataset(synthetic_df)
metadata          = Metadata(original_dataset)

profile = SyntheticDataProfile(
    original_dataset,
    synthetic_dataset,
    metadata=metadata,
    target='defaulted',
)

profile.generate_report(output_path='output/credit_quality_report.pdf')
print('Quality report saved → output/credit_quality_report.pdf')

## 4. Solution

### Question 1 — Does the synthetic dataset present major quality issues?

Yes — three clear issues, each in a different dimension.

#### Issue 1 — Class imbalance in `defaulted` (Utility)

The target column has a severe class imbalance problem in the synthetic data. The original dataset has ~16% defaults; the synthetic version has ~52%. A model trained on this synthetic data would learn a completely different prior for the target variable.

In [ ]:
print('Target distribution — defaulted:')
dist = pd.concat([
    original_df['defaulted'].value_counts(normalize=True).rename('original %'),
    synthetic_df['defaulted'].value_counts(normalize=True).rename('synthetic %'),
], axis=1).mul(100).round(1)
print(dist.to_string())

#### Issue 2 — Broken relationship between `age` and `age_group` (Fidelity)

`age_group` is a deterministic function of `age` — it is a binned version of the same variable. Generative models are probabilistic and do not understand deterministic rules, so they learn the relationship approximately and occasionally violate it. The fix is to remove `age_group` from the training data and recompute it as a calculated feature after generation.

In [ ]:
# Show records where age_group does not match age in the synthetic data
def expected_age_group(age):
    if age < 30:   return 'young'
    elif age < 50: return 'middle'
    else:          return 'senior'

synthetic_df['expected_age_group'] = synthetic_df['age'].apply(expected_age_group)
violations = synthetic_df[synthetic_df['age_group'] != synthetic_df['expected_age_group']]

print(f'Records where age_group does not match age: {len(violations):,} '
      f'({len(violations)/len(synthetic_df)*100:.1f}%)')
violations[['age', 'age_group', 'expected_age_group']].head(10)

#### Issue 3 — Quasi-identifiers: `age`, `income`, `zip_prefix` (Privacy)

The dataset contains three quasi-identifiers. Given that utility is our priority and the privacy score from the quality report is within an acceptable range, we apply a targeted intervention rather than strong privacy controls that would hurt utility.

In [ ]:
# zip_prefix carries the highest re-identification risk — show its uniqueness
n_unique_zip  = original_df['zip_prefix'].nunique()
n_total       = len(original_df)
print(f'zip_prefix: {n_unique_zip} unique values in {n_total:,} records '
      f'({n_unique_zip/n_total*100:.1f}% uniqueness)')

### Question 2 — Which metric dimension do you prioritise?

**Utility** — the synthetic data is intended for training a credit risk ML model. If the target distribution is wrong or feature relationships are broken, the model learns the wrong patterns and the data is unusable regardless of its fidelity or privacy scores. Privacy still has a governance floor, but the primary objective is predictive validity.

### Question 3 — What improvements would you recommend?

One fix per issue, in order of priority.

#### Fix 1 — Class imbalance: conditional training

The preferred fix is to retrain the synthesizer with `target` specified, which enables conditional generation and allows the synthesizer to respect the original class distribution. A post-hoc alternative is to undersample the overrepresented class in the synthetic output — it works but discards data without fixing the underlying generation problem.

In [ ]:
# Post-hoc fix: undersample the majority class to restore the original ratio
target_rate = (original_df['defaulted'] == 1).mean()
n_minority  = (synthetic_df['defaulted'] == 1).sum()
n_majority_target = int(n_minority / target_rate * (1 - target_rate))

majority = synthetic_df[synthetic_df['defaulted'] == 0].sample(n_majority_target, random_state=42)
minority = synthetic_df[synthetic_df['defaulted'] == 1]
rebalanced_df = pd.concat([majority, minority]).sample(frac=1, random_state=42).reset_index(drop=True)

print('After rebalancing:')
print(rebalanced_df['defaulted'].value_counts(normalize=True).mul(100).round(1).to_string())

#### Fix 2 — Broken relationship: treat `age_group` as a calculated feature

Remove `age_group` from the training data entirely and recompute it deterministically from `age` after generation. This is passed to the synthesizer as a `calculated_features` definition — the same pattern used in Chapter 2.

In [ ]:
# Define age_group as a calculated feature — pass this to RegularSynthesizer.fit()
def compute_age_group(age):
    return pd.cut(
        age,
        bins=[0, 29, 49, 120],
        labels=['young', 'middle', 'senior']
    ).astype(str).values

calculated_features = [
    {
        'calculated_features': 'age_group',
        'function': compute_age_group,
        'calculated_from': ['age'],
    }
]

# The synthesizer trains without age_group and enforces the rule at generation time
# synth.fit(X=dataset, metadata=metadata, calculated_features=calculated_features)
print('calculated_features defined ✓')
print('age_group will be computed from age at generation time — no probabilistic approximation.')

#### Fix 3 — Quasi-identifiers: anonymize `zip_prefix`

`zip_prefix` carries the highest re-identification risk due to its uniqueness. We anonymize it before training using a faker method. The privacy score for `age` and `income` was within the acceptable range, so we leave them as-is — dropping `age` in favour of `age_group` alone is an option if the privacy score worsens after retraining.

In [ ]:
# Pass this anonymize config to RegularSynthesizer.fit()
anonymize = {
    'zip_prefix': {'faker': 'postcode'},
}

# synth.fit(X=dataset, metadata=metadata, anonymize=anonymize, calculated_features=calculated_features)
print('anonymize config defined ✓')
print('zip_prefix will be replaced with a faker postcode before training.')